# RO装置データ解析（公開版）

## データについて（重要）
本プロジェクトは医療施設の運転データに基づく分析設計を示す目的で作成しています。
守秘義務の観点から **実データは公開せず**、GitHub公開版では **統計的性質を保ったダミーデータ**（`data/sample_data.csv`）に差し替えています。

- 実データはリポジトリに含めません
- 解析手法・設計・結果の読み方を再現できる形で公開します


In [ ]:
import pandas as pd

# 公開版（ダミーデータ）
DATA_PATH = "data/sample_data.csv"

# Excel互換を考慮して utf-8-sig を優先（BOM付きUTF-8でも読める）
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.head()


透過水水量(RO水)は30L/minで作成されていて、その作成量に合わせてポンプ周波数が変動して、RO膜入出口圧力が変動する。つまりRO膜を長持ちさせたいのであれば、透過水水量を最低限の必要量である25L/minまで下げてあげればRO膜は長持ちできるのではないかと考えた。東レは確かにRO膜を長持ちさせることはできるかもしれないが電導度が上昇する可能性があると言われた。

00_05.avif

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/sample_data.csv')
df.head(3)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
!pip install -q japanize_matplotlib

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

In [ ]:
# 可視化のサイズをノートブック上で統一する
plt.rcParams['figure.figsize'] = 10, 6

In [ ]:
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('bmh')

In [ ]:
df.describe(include='all')

In [ ]:
df['1段目透過水水量 (L/min)'].describe()

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

plt.figure(figsize=(14,6))
sns.boxplot(data=df[numeric_cols])
plt.xticks(rotation=90)
plt.title("数値変数の箱ひげ図")
plt.show()

In [ ]:
plt.figure(figsize=(20, 20))
df.hist(figsize=(30, 30))
plt.tight_layout()
plt.show()


In [ ]:
# 日付をdatetime型に変換
df['日付'] = pd.to_datetime(df['日付'])

# 日付でソート（念のため）
df = df.sort_values('日付')

# 日付をインデックスに設定
df = df.set_index('日付')

df.head()

In [ ]:
plt.figure()
plt.plot(df['1段目透過水水量 (L/min)'])
plt.title('透過水量の推移')
plt.xlabel('日付')
plt.ylabel('透過水量(L/min)')
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
df['1段目透過水水量 (L/min)'].plot(label='raw')
df['1段目透過水水量 (L/min)'].rolling(30).mean().plot(label='30日移動平均')
plt.title('透過水量（30日移動平均）')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
plt.plot(df['1段目透過水水質 (mS/m)'])
plt.title('透過水水質の推移')
plt.xlabel('日付')
plt.ylabel('透過水水質(mS/m)')
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
df['1段目透過水水質 (mS/m)'].plot(label='raw')
df['1段目透過水水質 (mS/m)'].rolling(30).mean().plot(label='30日移動平均')
plt.title('透過水水質（30日移動平均）')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

result = seasonal_decompose(df['1段目透過水水量 (L/min)'], model='additive', period=365)

result.plot()
plt.show()

- 透過水水量の時系列データの解釈
  - 観測値（Observed）
    - 初期（2011年前後）に一瞬低下 → すぐに30付近へ復帰
    - その後は小さな瞬間的ドロップはあるが、構造的な変化なし
  - トレンド（Trend）
    - 初期に立ち上がり、その後は横ばい〜ごく緩やかな変動
    - 右肩下がりではない
  
  「RO膜の目詰まりによる透過量低下」は示唆されない

  - 季節成分（Seasonal）
    - ±0.2〜0.3 L/min 程度の微小変動
    - 周期は明確だが振幅は小さい
  
  原水温,使用状況などによる運転条件依存の揺らぎ → 制御できている範囲

  - 残差（Resid）
    - ほぼランダム
    - 外れ値はあるが頻発しない

  異常兆候ではない

「長期間にわたり量的性能が非常に安定しており、交換を急がせるデータ的根拠は見当たらない」

In [ ]:
result = seasonal_decompose(df['1段目透過水水質 (mS/m)'], model='additive', period=365)

result.plot()
plt.show()

- 1段目透過水水質（mS/m）の解釈
  - 観測値（Observed）
    - 平均 0.17〜0.20 mS/m
    - 初期にスパイク（0.4〜0.5）があるが、その後は落ち着く
  
  初期スパイクは初期馴染み？ → 経年評価には使わない

  - トレンド（Trend）
    - 流れ
      1. 2011～2015 → わずかに低下（改善）
      2. 2016～2019 → ゆるやかに上昇（劣化方向）
      3. 2020～2023 → 一時的に低下
      4. 2024～ → 再び上昇傾向
    
  ジグザグだが、単調悪化ではない
    - 時系列を一本の因果ストーリー
      1. 2018年 管理者交代 → 運転・洗浄の質変更 → ポンプ周波数↑
      2. 対策として原水温度を上昇 → 透過水量は安定 → 水質は温度影響で上昇
      3. 2022年 給湯器故障 → 原水温度低下 → 水質が一時的に改善
      4. 復旧後 → 再び温度影響が乗り、水質上昇

  - 季節成分（Seasonal）
    - 非常に明確な周期
    - 振幅は ±0.002 mS/m 程度と小さいが、形が揃っている
      - 原水温の季節変動
      - 電気伝導度の温度依存性
  
  健全な季節性

  - 残差（Resid）
    - ほぼ0中心
    - 分散は小さい
    - 構造的な偏りなし
  
  異常イベントは局所的

水質は運転条件に応じて可逆的に揺れているが、RO膜の致命的劣化を示す一貫した悪化トレンドはない

このRO膜は、量的性能は全期間を通して安定しており、水質変動も運転条件依存の範囲に収まっている。少なくともデータ上は交換を急ぐ根拠は見当たらないと考えられる。
  

In [ ]:
sns.pairplot(df);

In [ ]:
# x: 透過水水量（RO水量）
x = df["1段目透過水水量 (L/min)"]

# y: 透過水水質（EC）
y = df["TF"]

sns.scatterplot(x=x, y=y, alpha=0.3)

plt.xlabel("透過水水量 (L/min)")
plt.ylabel("TF有無")
plt.title("透過水水量とTFの関係")

plt.tight_layout()
plt.show()

In [ ]:
# x: 透過水水量（RO水量）
x = df["1段目透過水水量 (L/min)"]

# y: 透過水水質（EC）
y = df["1段目透過水水質 (mS/m)"]

sns.scatterplot(x=x, y=y, alpha=0.3)

plt.xlabel("透過水水量 (L/min)")
plt.ylabel("透過水水質 EC (mS/m)")
plt.title("透過水水量と透過水水質の関係")

plt.tight_layout()
plt.show()

- 透過水水量は「ほぼ2つの塊」
  - 約25 L/min 周辺
  - 約30 L/min 周辺
  
運転設定がほぼ固定で、たまに別設定（25前後）があった
という現場の運転実態を、そのまま反映している。

- 透過水水質の分布
  - 30L/min帯 → 透過水水質は約0.14〜0.22mS/mに集中
  - 25L/min帯 → 透過水水質は約0.17〜0.25mS/m程度

25L/minの方が「やや高めに見えるが、大きな跳ね上がりはない」

- 見られないもの
  - 明確な右上がり（=水量を上げるとECが必ず上がる）
  - 明確な右下がり（=水量を下げるとECが必ず上がる）

**単純な線形関係は見えない**

**透過水水量は「自然に変動した結果」ではなく、制御目標として意図的に固定・調整されている変数だから**

- 見えるもの
  - 透過水水量を30→25にしても透過水水質が破綻的に上がる様子はない
  - 透過水水質のばらつきは透過水水量よりも他の因子に支配されていそう

- 東レの「透過水水質が上がるかも」という話との関係
  - 東レの言っていることは：
  
  理論的には
  
  流量低下 → 膜内流速低下 → 透過水水質上昇の可能性

- 実データでは
  - 25L/minでも透過水水質は管理可能な範囲臨床的に危険なレベルへの上昇は見えない

In [ ]:
plt.figure()
sns.heatmap(df.corr(method='pearson'),
            cmap='coolwarm',
            annot=True,        # ← 相関係数を表示
            fmt='.2f',         # ← 小数点2桁で表示
            square=True)
plt.title('ピアソン相関')
plt.show()


In [ ]:
plt.figure()
sns.heatmap(df.corr(method='spearman'),
            cmap='coolwarm',
            annot=True,
            fmt='.2f',
            square=True)
plt.title('スピアマン相関')
plt.show()


- 相関係数(透過水水量×透過水水質)
  - ピアソン＆スピアマンともに相関係数は低め
  - 透過水水量は「制御目標」であり、自然変動の結果ではないため、相関が観測されにくい

- 相関係数(圧力×ポンプ周波数)
  - 相関≈約0.9

  完全に制御系
  
  ポンプ周波数で圧力を作っている→これは正常

- 相関係数(回収率×流量(濃縮排水・リターン))
  - 強い負の相関

  水量バランス制御

- 相関係数(原水温度×透過水温度)
  - 相関≈約0.99

  物理的に当然

- 相関係数(原水温度×透過水水質)
  - 中程度の正相関

  膜透過特性の温度依存、これは透過水水量よりも透過水水質に直接効いている因子か？

- 相関関係(透過水水量×TF)
  - 点双列相関になっている

  TFと透過水量の間に高いピアソン相関が認められたが、TFは二値変数であり、これはTF有無による水量分布の差を反映したもので、連続的な線形関係を示すものではない。

- 透過水水量について
  - 水量は
    - 圧力
    - ポンプ周波数
    - 回収率

  と強く結びついている。

  - しかし
    - 透過水水質との単純相関は弱い

- 相関だけで言える「慎重な結論」相関係数ベースで、正しく・安全に言えることは次の通りです。
  - 言えること
    - 透過水水量と透過水水質の単純相関は弱い
    - よって「水量を下げたら必ず水質が上がる」とは言えない
    - 透過水水質は温度条件との関係が強い
  - 言えないこと
    - 透過水水量を25L/minにしても透過水水質は大丈夫
    - 透過水水量を下げると透過水水質が悪化する
  
  どちらも相関だけでは判断不可

- 相関考察

透過水水量と透過水水質との間に強い単純相関は認められなかった。
これは透過水水量が制御目標として維持されている変数であり、
原水条件や膜特性の変化に対してポンプ周波数・圧力調整により補償されているためと考えられる。
したがって、透過水水量と透過水水質の因果関係を評価するには、他の運転条件を考慮した多変量解析が必要であると考えられる。

In [ ]:
def detect_anomaly_sudden_change(series, window=7, threshold=3):
    """
    series : 時系列（例：df['1段目ROモジュール差圧(MPa)']）
    window : 移動平均の窓（日数）
    threshold : 何σ以上を異常とするか
    """
    # 移動平均（中心をそろえるため center=True）
    ma = series.rolling(window=window, center=True).mean()

    # 移動平均からの絶対乖離
    diff = (series - ma).abs()

    # 乖離のしきい値（平均＋threshold×標準偏差）
    limit = diff.mean() + threshold * diff.std()

    # 異常フラグ True/False
    anomaly_flag = diff > limit

    return ma, diff, anomaly_flag


In [ ]:
def plot_anomaly(series, anomaly_flag, ma=None, title=None):
    plt.figure(figsize=(12,4))

    # 元の値
    plt.plot(series.index, series.values, label="value")

    # 移動平均も表示したい場合
    if ma is not None:
        plt.plot(ma.index, ma.values, linestyle="--", label="moving avg")

    # 異常点（赤）
    plt.scatter(series.index[anomaly_flag],
                series[anomaly_flag],
                color="red", label="anomaly", zorder=3)

    plt.grid(True)
    plt.legend()
    if title:
        plt.title(title)
    plt.xlabel("日付")
    plt.ylabel(series.name)
    plt.show()


In [ ]:
target_cols = [
    '1段目透過水水質 (mS/m)',
    '1段目透過水水量 (L/min)'
]

for col in target_cols:
    ma, diff, anom = detect_anomaly_sudden_change(df[col], window=7, threshold=3)
    print(col, "異常値 個数:", anom.sum())
    plot_anomaly(df[col], anom, ma=ma, title=f"{col} の異常検知（急変）")

# 重回帰分析

- なぜ単純な相関ではダメか
  - 原水温度が高いときは → 運転条件も同時に変わっている
  - リターン水量・濃縮水排出量・ポンプ周波数は → 互いに関連して動く

  相関だけでは「どれが本当に効いているか」分からない

そこで、複数の因子を同時に考慮し、それぞれの影響を切り分けて評価するために重回帰分析（OLS）を用いた

- 透過水水量が、どの運転条件・設定因子によって決まっているのかを、他の条件を同時に考慮した上で整理するために透過水水量を目的変数とした重回帰分析を行うこととする。

- なぜ目的変数を「透過水水量」にしたのか
  - 透過水水量は人為的に設定・制御される量
  - メーカー説明でも
    - リターン水量
    - 濃縮水排出量
    
  によって決まるとされている

  - さらに、水質は水量の結果として変化する可能性がある

まず「水量を決めている因子」を明らかにしないと、水質の議論ができない

そのため 第1段階の目的変数として透過水水量を選択した。

- なぜこの説明変数を選んだのか
  - 原水温度（物理条件）
    - 水の粘度・膜透過性に影響
    - 季節変動を代表する物理因子
    - 装置外要因として必須
  - リターン水量（操作レバー）
    - 人為的に設定できる
    - メーカーが「水量を決める」と説明している
    - 最も直接的な制御因子
  - 濃縮水排出量（操作レバー）
    - リターン水量と対になる設定値
    - 回収率・膜負荷を反映
    - 水量を下げる方向に働く因子
  - ポンプ周波数（駆動条件）
    - 圧力・流量の上流側の制御量
    - 「機械的な押し出しの強さ」
    - 運転強度の代表
  - 1段目RO差圧（膜状態）
    - 膜の抵抗・目詰まり状態を表す
    - 操作変数ではないが、背景条件として重要
    - 長期劣化・異常検知の意味を持つ

物理条件・人為的設定・駆動条件・膜状態をそれぞれ代表する変数を1つずつ選び、水量に対する影響を整理した

- TFを入れた理由
  - TFの有無で水量の分布が明らかに異なるTFを入れないと → 「運転モードの違い」を他の係数が代わりに説明してしまう。

TFは原因というより運転条件の違いを補正するための調整変数

In [ ]:
import statsmodels.api as sm

y = df["1段目透過水水量 (L/min)"]

X = df[[
    "原水温度 (℃)",
    "1段目ﾘﾀｰﾝ水量 (L/min)",
    "濃縮水排水量 (L/min)",
    "ﾎﾟﾝﾌﾟ周波数(P2) (Hz)",
    "1段目ROﾓｼﾞｭｰﾙ差圧 (MPa)",
    "TF"
]]

X = sm.add_constant(X)

# 時系列なので標準誤差をHAC(Newey-West)で補正
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":7})
print(model.summary())


- モデル全体の当てはまり（R²）
  - R² = 0.519（調整済みも同等）
    - 透過水水量の変動の約52%をこのモデルが説明している
    - 残り約48%は,日内変動,微細な制御,未測定因子,ノイズ？
- F検定（モデル全体の有意性）
  - F-statistic : 有意
  - Prob(F-statistic)：極めて小さい

明確に意味がある。少なくとも1つ以上の説明変数は水量に本質的に関係している

- Durbin-Watson（自己相関）
  - 回帰で説明しきれなかった残差が、時間的に独立かどうか？
    - 膜汚れは1日で消えない
    - 微妙な調整ズレは翌日も残る
    - 運転方針は急に変わらない
  
  説明できなかった48%の中に「昨日の影響を引きずる成分」が含まれている

- Cond. No.
  - 説明変数同士が、どれくらい似た動きをしているか？
    - 小さい → 独立
    - 大きい → 強く連動
  
  今回は大きい

多重共線性が強い場合、係数の厳密な数値解釈は慎重に行う必要があるが、どの因子群が主要な役割を担っているかという構造的理解には十分有用である。
また、RO装置の運転構造やメーカーからの知見を踏まえ、物理的・運用的に不整合な解釈は排除しながら結果を評価している。

- 各説明変数の解釈
  - 原水温度（℃）
    - 係数 : -0.021
    - p = 0.009（有意）
      - 解釈
        - 原水温度が1℃上がると、透過水水量は約0.02L/min低下
        - 物理的には本来「温度↑ → 透過↑」が自然？
        - 多重共線性？
      
      
      温度そのものの物理効果ではなく、高温期に運転を絞っている「運用判断」が反映されている？
  - リターン水量（L/min）
    - 係数 : +0.102
    - p < 0.001（強く有意）
      - 解釈
        - リターン水量を1L/min増やすと、透過水水量は約0.10 L/min増加
        - メーカー説明・装置制御ロジックと完全一致
  - 濃縮水排出量（L/min）
    - 係数 : -0.268
    - p = 0.001（有意）
      - 解釈
        - 濃縮水排出量を 1 L/min増やすと、透過水水量は約0.27 L/min減少
        - 回収率を下げる方向の操作が、水量低下に直結
        - 水量を下げる最重要因子の一つ
  - ポンプ周波数（P2）（Hz）
    - 係数 : +0.0088
    - p = 0.35（有意でない）
      - 解釈
        - 周波数を上げても、他条件を同時に考慮すると水量への影響は限定的
        - 周波数はリターン水量,排出量によって「結果的に調整されている」可能性あり
        - 主制御因子ではなく補助因子
  - 1段目RO差圧（MPa）
    - 係数 : +3.52
    - p = 0.058（境界的）
      - 解釈
        - 統計的にやや不安定
        - 符号が「＋」なのは差圧が 駆動圧として働いている場面を拾っている可能性あり
        - 水量を直接決める因子というより、運転状態・膜状態を反映する背景変数？
  - TF（0/1）
    - 係数：+1.94
    - p = 0.002（明確に有意）
      - 解釈
        - TF=1の運転モードでは他の条件が同じでも、透過水水量が平均で約2L/min高い


今回、透過水水量が何によって決まっているのかを整理するために、
重回帰分析を行った。
目的変数を透過水水量、説明変数には原水温度、リターン水量、濃縮水排出量、ポンプ周波数、RO差圧を選んだ。
これは、物理条件・人為的設定・駆動条件・膜状態をそれぞれ代表する因子で、さらに、運転モードの違いを補正するためTFも加えた。
その結果、透過水水量は特にリターン水量、濃縮水排出量、TFを含めた運転モードの影響が大きいことが分かった。

# Lasso回帰

- ここで行うこと
  - どの説明変数が残るか？
  - OLSで考えた「主役因子群」が統計的にも不要ではないかの確認

OLSの裏取り（健全性チェック）です。

##Lasso回帰とは「予測精度を保ちつつ、重要でない説明変数の係数を0にしてくれる回帰」

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(
    cv=5,              # クロスバリデーション
    random_state=0,
    max_iter=10000
)

lasso.fit(X_scaled, y)

In [ ]:
coef = pd.Series(
    lasso.coef_,
    index=X.columns
)

coef

- Lasso回帰の解釈
  - すべての係数 = 0
  - この説明変数セットでは、単独で安定して効く変数はないと判断している？ = 因子は個別ではなくセット（因子群）として効いている
  - リターン水量・濃縮水排出量・TF・周波数 → 常に一緒に動く
  - Lasso回帰は「代表1個だけ残す or 全部消す」性質を持つ
  - 今回のように**全員セットで意味がある系のデータでは、全消しは典型的な挙動らしい。**

# Rigde回帰

- 符号（＋か −か）の確認
- 正か負かは比較的安定し多重共線性下でも信頼しやすい
- 特徴量の絶対値の大小ではなく「順位」をみる
- どの因子群が主役か？

In [ ]:
from sklearn.linear_model import RidgeCV

ridge = RidgeCV(
    alphas=np.logspace(-3, 3, 50),
    cv=5
)

ridge.fit(X_scaled, y)

In [ ]:
coef_ridge = pd.Series(
    ridge.coef_,
    index=X.columns
)

coef_ridge

| 変数     | 係数         | 解釈            |
| ------ | ---------- | ------------- |
| 原水温度   | **-0.047** | 温度↑ → 透過水量↓   |
| リターン水量 | **+0.155** | 設定上、増やすと透過水量↑ |
| 濃縮水排出量 | **-0.206** | 排出↑ → 透過水量↓   |
| ポンプ周波数 | **+0.022** | 影響は小さいが正      |
| RO差圧   | **+0.010** | ほぼ中立          |
| TF     | **+0.254** | 強い正の関係        |


- 解釈
  - TF（+0.254）
  - 濃縮水排出量（-0.206）
  - リターン水量（+0.155）

メーカーが言っていた構造と完全一致「リターン水量と濃縮水排出量の設定で透過水量を決める」


- 解釈
  - 原水温度が「負」
  - OLSでもRidgeでも一貫して「負」

温度が高いと、意図的に透過水量を絞っている（運転制御・膜保護・水質意識）
  

- 解釈
  - TF（0/1変数）の意味
  - TFが最も大きい正係数,これは:TF = 1 のとき透過水量が系統的に高い状態

「透過水水量を目的変数として、設定可能な運転因子と物理因子を用いてRidge回帰で分析した。

説明変数間に強い相関があるため単独因子の検定ではなく、因子群としての方向性を確認する目的である。

その結果、リターン水量・濃縮水排出量・TFといった設定因子が一貫して主役であり、原水温度は負の方向で制御されていることが確認された。」

# 水質を目的変数にした重回帰分析

- 今回やりたい検証「透過水水量を30→25に下げると、透過水水質は上昇するのか？」
- 操作可能な水量が、水質に因果的に効いているか？

In [ ]:
y = df["1段目透過水水質 (mS/m)"]

X = df[
    [
        "1段目透過水水量 (L/min)",
        "原水温度 (℃)",
        "1段目ROﾓｼﾞｭｰﾙ差圧 (MPa)",
        "TF"
    ]
]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 7})
print(model.summary())


- モデル全体の解釈
  - 決定係数 R²≈0.21
    - 水質というのは：
        - 微細な膜現象
        - 未観測要因（原水組成、瞬時流れ、履歴）
        - 非線形・遅延

      が強く効くため、

  OLSで 0.2 出ていれば「説明できている」部類？

  - F統計量:有意（p≪0.001）
    - モデル全体として意味のある説明力がある

- 解釈
  - 1段目透過水水量（L/min）
    - 係数:-0.0010
    - p値:0.246（有意ではない）

**このデータ範囲では「水量を下げると水質が改善する」と明確には言えない**




- 解釈
  - 原水温度（℃）
    - 係数:+0.0029
    - p値:< 0.001（有意）

原水温度が高いほど透過水水質は悪化,物理的にも非常に妥当,強い交絡因子？

- 解釈
  - 1段目RO差圧（MPa）
    - 係数:+0.0837
    - p値:0.224（有意ではない）

- 解釈
  - TF（0/1）
    - 係数:-0.0247
    - p値:< 0.001（有意）
    - 符号：負

**TF=1 の運転では水質が有意に良い**

**他の条件（温度・TF・差圧）を同時に考慮すると、水量単独で水質を悪化させる証拠は見えない**

**TFを含めた運転モード時に水質を強く規定している？**

**即時効果ではない可能性あり**
**水量 ↓ → 膜状態変化 → 水質悪化は時間遅れ があるはず。同時刻OLSでは検出できない**

# 主成分分析

In [ ]:
from sklearn.decomposition import PCA

features = [
    "1段目ﾘﾀｰﾝ水量 (L/min)",
    "濃縮水排水量 (L/min)",
    "原水温度 (℃)",
    "1段目ROﾓｼﾞｭｰﾙ差圧 (MPa)",
    "TF"
]

X = df[features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [ ]:
explained_var = pd.DataFrame(
    {
        "寄与率": pca.explained_variance_ratio_,
        "累積寄与率": pca.explained_variance_ratio_.cumsum()
    },
    index=[f"PC{i+1}" for i in range(len(features))]
)

explained_var

- 寄与率の確認
  - 第一主成分（PC1）が 0.333183、第二主成分（PC2）が0.303999です。これはつまり、第一主成分が元データの約33%、第二主成分が約30%表現できているという意味になる。
  - これらの値を足したものが累積寄与率ですので、今回の第三主成分までの累積寄与率は約82%です。目安として、第四主成分までの累積寄与率が70～80% であると、元々のデータを比較的上手く表現できていると評価する。
  - 第三主成分までの累積寄与率82.1%、70〜80%超で構造把握には十分、第三主成分まで表示・解釈する方針で進める。
  

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=[f"PC{i+1}" for i in range(len(features))]
)

loadings

In [ ]:
plt.figure(figsize=(8, 4))
sns.heatmap(
    loadings.iloc[:, :3],   # PC1〜PC3まで
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("主成分負荷量ヒートマップ（PC1〜PC3）")
plt.ylabel("変数")
plt.xlabel("主成分")
plt.tight_layout()
plt.show()


- 主成分負荷量ヒートマップ解釈
  - PCA1
    - 主に効いている変数：
      - 濃縮水排出量（＋）
      - TF（ー）
      - RO差圧（＋）
  
  TFを含めた装置全体の状態を考慮した軸？

  - PCA2
    - 主に効いている変数：
      - 原水温度（＋）
      - RO差圧（ー）
      - リターン水量（ー）
      - TF（ー）
  
  「原水温度・季節要因＋運転補正」の軸

  「2018年以降の温度調整対応」が、ここで数理的に裏付けられています。

  - PCA3
    - 主に効いている変数：
      - リターン水量（＋）
      - RO差圧（ー）

  「循環構造・内部フローの癖」

  これは、透過水量、水質に遅れて効いてくる可能性が高い因子群？

  「2周目（ラグ分析）」で主役になる主成分？


RO装置は「単一の因子」で動いているのではなく、状態・環境・構造という複数の因子の重ね合わせで振る舞っている

主成分分析は因果関係を示すものではない。
主成分分析で因子構造の存在が示唆されたため、次に因子分析を用いて装置状態・温度要因・循環構造という潜在因子が実際にデータから支持されるかを確認する。

- 今の流れ
  - 1周目：OLS → 何の変数が効いてそうか？

  - PCA：→ 構造はどうなっているか？

  - 因子分析：→ 本当の原因（潜在因子）は何か？


# 因子分析

PCAで示唆された「装置状態」「原水温度（季節）」「循環構造」という 潜在因子 が本当に存在するかを検証

In [ ]:
cols = [
    "1段目ﾘﾀｰﾝ水量 (L/min)",
    "濃縮水排水量 (L/min)",
    "原水温度 (℃)",
    "1段目ROﾓｼﾞｭｰﾙ差圧 (MPa)",
    "TF"
]

In [ ]:
X = df[cols].copy()

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

In [ ]:
corr = np.corrcoef(X_std, rowvar=False)
eigenvalues, _ = np.linalg.eig(corr)

eigenvalues

In [ ]:
plt.plot(range(1, len(eigenvalues)+1), eigenvalues, marker='o')
plt.axhline(1, color='red', linestyle='--')
plt.xlabel("因子番号")
plt.ylabel("固有値")
plt.title("Scree Plot")
plt.show()

カイザー基準では「2因子」 が妥当

In [ ]:
!pip install factor-analyzer

In [ ]:
from factor_analyzer import FactorAnalyzer

fa = FactorAnalyzer(
    n_factors=2,
    rotation="varimax",
    method="ml"
)

fa.fit(X_std)

In [ ]:
loadings = pd.DataFrame(
    fa.loadings_,
    index=cols,
    columns=["Factor1", "Factor2"]
)

loadings

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(
    loadings,
    annot=True,
    cmap="coolwarm",
    center=0
)
plt.title("因子負荷量ヒートマップ")
plt.show()

In [ ]:
factor_scores = pd.DataFrame(
    fa.transform(X_std),
    columns=["Factor1", "Factor2"],
    index=df.index
)

factor_scores.head()

### 第一因子

| 変数     | 因子負荷量     |
| ------ | --------- |
| TF     | **+0.97** |
| 濃縮水排水量 | **−0.71** |
| その他    | 小さい       |


- TFの有無が圧倒的に支配的
    - RO膜に入る水の質と負荷を決める潜在因子

Factor1 = ROに入る前の水の状態・負荷を規定する支配因子

### 第二因子

| 変数      | 因子負荷量     |
| ------- | --------- |
| 原水温度    | **−0.64** |
| 1段目RO差圧 | **+0.47** |
| リターン水量  | +0.38     |
| TF      | +0.25     |


- Factor2 は水の流れと季節要因？
  - 原水温度（季節変動）
  - 差圧（膜負荷）
  - 循環構造（リターン水）

Factor2 = 流体力学？＋季節変動？＋膜負荷因子？

Factor2 は水の流れと季節要因？

# **当院のRO装置は、ROに入る前の水の状態、季節変動による原水温度、水の循環構造、流れの癖で表されている？**

データサイエンスサイクル1周目で即時効果は弱いことを確認し、2周目では状態変化の時間遅れを検証する

# 次の仮説検証は「時系列順で考えた時の水量↓のあとに、水質が悪化する」仮説を検証する。

In [ ]:
max_lag = 30  # 最大30日遅れまで見る
lags = range(0, max_lag + 1)

corrs = []
for lag in lags:
    corr = df["1段目透過水水質 (mS/m)"].corr(
        df["1段目透過水水量 (L/min)"].shift(lag)
    )
    corrs.append(corr)

plt.figure(figsize=(10,4))
plt.plot(lags, corrs, marker='o')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel("ラグ（日）")
plt.ylabel("相関係数")
plt.title("水量 → 水質 のラグ相関")
plt.show()


- 解釈
  - lag = 0（日）
    - 相関係数は約−0.23（最も強い）
  - lag が増えるにつれて
    - 相関の絶対値が小さくなり
    - lag15日あたりで段差的に弱まる
  - lag20〜30日
    - 相関は−0.12〜−0.14程度で横ばい
  
- 強いのは「即時〜数日以内」
- 1〜2週間を境に、関係性の強さが変わる

- 解釈
  - 符号が「負」で一貫している意味
    - 水量が多い→水質指標は低下
  - 即時効果が最も強い
  - 中長期の影響が残っている
- 水量と水質には一貫した負の関係
- 影響は即時＋中期的に持続

In [ ]:
# 代表的なラグを作成
for lag in [1, 7, 14]:
    df[f'water_volume_lag{lag}'] = df["1段目透過水水量 (L/min)"].shift(lag)

# 欠損を除外
df_lag = df.dropna()

In [ ]:
import statsmodels.api as sm

X = df_lag[
    ['1段目透過水水量 (L/min)',        # lag0（当日）
     'water_volume_lag1',
     'water_volume_lag7',
     'water_volume_lag14']
]

X = sm.add_constant(X)
y = df_lag['1段目透過水水質 (mS/m)']

model = sm.OLS(y, X).fit()
print(model.summary())

- 回帰係数と p値の読み取り
  - 当日水量（lag0）
    - 係数：−0.0023
    - p値：< 0.001（有意）
    - 水量が下がると、その日のうちに水質が悪化する傾向が明確に存在

  - water_volume_lag1
    - 係数：−0.0030
    - p値：< 0.001
    - 前日の水量も、当日の水質に独立して影響
  
  - water_volume_lag7
    - 係数：−0.0012
    - p値：0.022（有意）
    - 1週間前の水量が、当日の水質に影響

  - water_volume_lag14
    - 係数：−0.0010
    - p値：0.044（ギリギリ有意）
    - 2週間前の水量も、弱いが独立した影響




In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif["variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i)
              for i in range(X.shape[1])]

vif

- VIF（多重共線性）の評価
| 変数    | VIF  |
| ----- | ---- |
| 当日水量  | 2.08 |
| lag1  | 2.12 |
| lag7  | 2.03 |
| lag14 | 1.72 |
  - すべて VIF < 3
  - 多重共線性は問題なし
  - 各ラグは「意味の異なる情報」を持っている


- モデル全体の評価
  - R² ≈ 0.064
  - F-test p値 ≪ 0.001
    - モデル全体は有意

- 水量を下げて、水質が悪くなるのは即時性の可能性がある？
- 即時性が最も強いが、それだけではなく、水量は装置の状態として 1〜2週間影響を残す

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

# 対象データ（順番が超重要）
data = df[['1段目透過水水質 (mS/m)', '1段目透過水水量 (L/min)']].dropna()

In [ ]:
max_lag = 14  # Step2と整合させる

grangercausalitytests(
    data,
    maxlag=max_lag,
    verbose=True
)

- 結論
  - 「水量 → 水質」の時間的因果性は強く支持されます。しかもその有効な時間幅は：1日後〜13日後まで有意、14日後では消失です。

  - 有意だったラグ
    - lag1〜13：p < 0.05
      - 特にlag1〜5 は p ≪ 0.001
      - lag12,13もギリギリ有意
  
  - 非有意
    - lag14：p = 0.129（非有意）
      - 水量の過去情報は最大で約2週間弱（〜13日）
      - それ以降は水質予測に寄与しない

- Step2（回帰）との整合性
  - Step2（ラグ付き回帰）
    - lag0：有意
    - lag1：有意
    - lag7：有意
    - lag14：ギリギリ有意
  - Step3（グレンジャー因果）
    - lag1〜13：有意
    - lag14：非有意
- 水量は即時に水質へ影響し、装置状態として 約2週間以内の時間スケールで影響を持つ

In [ ]:
# 逆方向（水質 → 水量）
data_rev = df[['1段目透過水水量 (L/min)', '1段目透過水水質 (mS/m)']].dropna()

grangercausalitytests(
    data_rev,
    maxlag=max_lag,
    verbose=True
)

- 水量→水質の結果との非対称性
| 観点   | 水量 → 水質   | 水質 → 水量 |
| ---- | --------- | ------- |
| 有意ラグ | 1〜13日（連続） | 断続的     |
| p値   | 極めて小さい    | 境界的〜中程度 |
| 物理説明 | 非常に自然     | 操作介入が必要 |
| 解釈   | 状態変数      | 制御応答    |


グレンジャー因果性検定の結果、水量の過去情報は水質の予測精度を一貫して有意に向上させた。一方、水質から水量への因果性は一部のラグで有意性を示したものの、その効果は不連続かつ限定的であり、主として運転操作に起因するフィードバックの影響と考えられた。以上より、時間的因果の主方向は水量から水質である可能性が高い。

In [ ]:
# 年間周期
df['dayofyear'] = df.index.dayofyear
df['sin_year'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['cos_year'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

In [ ]:
df['water_volume_lag1']  = df['1段目透過水水量 (L/min)'].shift(1)
df['water_volume_lag7']  = df['1段目透過水水量 (L/min)'].shift(7)
df['water_volume_lag14'] = df['1段目透過水水量 (L/min)'].shift(14)

df_lag = df.dropna()

In [ ]:
X = df_lag[
    [
        '1段目透過水水量 (L/min)',
        'water_volume_lag1',
        'water_volume_lag7',
        'water_volume_lag14',
        '原水温度 (℃)',
        'sin_year',
        'cos_year'
    ]
]

X = sm.add_constant(X)
y = df_lag['1段目透過水水質 (mS/m)']

model_step4 = sm.OLS(y, X).fit()
print(model_step4.summary())


- 交絡調整付き OLS の解釈
  - 目的
    - 「水量 → 水質」の関係が季節（年周期）、原水温度を調整してもなお残るか を確認する。

- 解釈
  - 水量（当日）
    - coef = -0.0015
    - p = 0.002（有意）
  - 水量が下がると、同日に水質は悪化する（即時効果）


| 変数                 | 有意性    | 解釈            |
| ------------------ | ------ | ------------- |
| water_volume_lag1  | **有意** | 前日の水量低下が水質に影響 |
| water_volume_lag7  | 非有意    | 週単位の影響は弱い     |
| water_volume_lag14 | 非有意    | 2週間後効果はほぼなし   |

- 解釈
  - 原水温度
    - coef = 0.004
    - p < 0.001（強く有意）
  - 温度 ↑ → 透過性 ↑ → 電導度変化

  - sin_year / cos_year
    - 両方とも有意
    - 「原水温度だけでは説明しきれない年周期要因（雨量・水源・原水成分）」が存在することを意味します。
    - 水量は水質の予測因子であり、時系列的にも先行している。

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

resid = model_hac14.resid

plt.figure()
plot_acf(resid, lags=60)
plt.show()

plt.figure()
plot_pacf(resid, lags=60)
plt.show()


In [ ]:
X = df_lag[
    ['1段目透過水水量 (L/min)',
     'water_volume_lag1','water_volume_lag7','water_volume_lag14',
     '原水温度 (℃)','sin_year','cos_year']
].dropna()

y = df_lag.loc[X.index, '1段目透過水水質 (mS/m)']

X = sm.add_constant(X)

# HACのラグ長：まずは14（2週間）か 30（1ヶ月）で試すのが無難
model_hac14 = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':14})
print(model_hac14.summary())

model_hac30 = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':30})
print(model_hac30.summary())


In [ ]:
for c in df.columns:
    print(repr(c))